# Capital Raise Detector

Predict the likelihood of companies performing primary equity raises based on 5 financial signals.

## Overview

This notebook analyzes companies on:
1. **Cash Runway** - How many months of cash remains
2. **Liquidity Stress** - Current/quick ratios and working capital
3. **Debt Maturity** - Near-term debt obligations
4. **Operational Red Flags** - Revenue trends, margins, FCF
5. **Market & Behavioral Signals** - Stock price, insider activity

Companies scoring > 50 are flagged for potential capital raise activity.

In [ ]:
# Import required modules
from analyzer import CapitalRaiseAnalyzer
from data import EXAMPLE_COMPANIES, EARNINGS_CALL_TRANSCRIPT, MARKET_NEWS
from models import FinancialMetrics
import pandas as pd
from IPython.display import display, Markdown


## Initialize Analyzer

Create analyzer instance with your preferred LLM model.

In [ ]:
# Initialize with default model (gpt-4o-mini)
# For higher accuracy, use 'gpt-4' or 'gpt-4o'
analyzer = CapitalRaiseAnalyzer(model_name="gpt-4o-mini")
print("✓ Analyzer initialized")

## Analyze Example Companies

Run the detector on our 5 example companies with different risk profiles.

In [ ]:
# Analyze all example companies
predictions = analyzer.batch_analyze(EXAMPLE_COMPANIES)

# Display summary
summary_data = [
    {
        "Ticker": p.ticker,
        "Company": p.company_name,
        "Score": f"{p.likelihood_score:.1f}",
        "Risk Level": p.risk_level.upper(),
        "Alert": "⚠️  YES" if p.above_threshold else "✓ No",
        "Confidence": f"{p.confidence:.0f}%",
    }
    for p in predictions
]

summary_df = pd.DataFrame(summary_data)
display(Markdown("### Summary of All Companies"))
display(summary_df)

## Detailed Analysis by Company

Inspect individual company scores and key drivers.

In [ ]:
# Show detailed results for each company
for pred in predictions:
    print(pred)

## Signal Breakdown

Visualize the contribution of each signal to the overall score.

In [ ]:
# Create signal breakdown for each company
signal_data = []
for p in predictions:
    signal_data.append({
        "Ticker": p.ticker,
        "Cash Runway (0-40)": f"{p.signal_scores.cash_runway_score:.1f}",
        "Liquidity (0-20)": f"{p.signal_scores.liquidity_stress_score:.1f}",
        "Debt Maturity (0-15)": f"{p.signal_scores.debt_maturity_score:.1f}",
        "Operational (0-15)": f"{p.signal_scores.operational_red_flags_score:.1f}",
        "Market (0-10)": f"{p.signal_scores.market_behavioral_score:.1f}",
        "Total (0-100)": f"{p.likelihood_score:.1f}",
    })

signals_df = pd.DataFrame(signal_data)
display(Markdown("### Signal Score Breakdown"))
display(signals_df)

## Alerts

Companies scoring above 50 (threshold) that need monitoring.

In [ ]:
# Get companies above threshold
alerts = analyzer.get_alerts(predictions)

if alerts:
    display(Markdown(f"### ⚠️  {len(alerts)} Companies Above Threshold"))
    for alert in alerts:
        print(f"\n**{alert.ticker}** - Score: {alert.likelihood_score:.1f}")
        print(f"Risk Level: {alert.risk_level.upper()}")
        print("Key Drivers:")
        for driver in alert.key_drivers:
            print(f"  • {driver}")
else:
    print("No companies above threshold.")

## With Market Signal Analysis

Include LLM analysis of earnings calls and news (requires OpenAI API key).

In [ ]:
# Re-analyze with market signals
# This uses the LLM to analyze earnings call transcripts and news
# WARNING: This makes API calls to OpenAI

print("Analyzing company with market signals...")
enhanced_pred = analyzer.analyze(
    EXAMPLE_COMPANIES[1],  # BURN company
    earnings_call_transcript=EARNINGS_CALL_TRANSCRIPT,
    market_news=MARKET_NEWS,
)

print(enhanced_pred)
print(f"\nMarket signals added: {len(enhanced_pred.key_drivers)} key drivers identified")

## Custom Company Analysis

Analyze your own company data.

In [ ]:
# Example: Create custom company
from models import FinancialMetrics

my_company = FinancialMetrics(
    ticker="MYCORP",
    company_name="My Custom Company",
    cash_and_equivalents=100_000_000,
    monthly_burn_rate=5_000_000,
    current_assets=150_000_000,
    current_liabilities=80_000_000,
    quick_assets=120_000_000,
    accounts_payable=30_000_000,
    total_debt=50_000_000,
    debt_due_12mo=10_000_000,
    debt_due_6_18mo=15_000_000,
    revenue_trailing_12m=500_000_000,
    revenue_prior_year=450_000_000,
    operating_cash_flow_trailing_12m=50_000_000,
    gross_margin=0.60,
    prior_gross_margin=0.58,
    capex_annual=20_000_000,
    stock_price=100.0,
    stock_price_52w_high=110.0,
    insider_selling_activity="normal",
    auditor_going_concern=False,
    credit_rating="BBB",
    credit_rating_outlook="stable",
)

# Analyze
custom_result = analyzer.analyze(my_company)
print(custom_result)

## Score Interpretation Guide

### Risk Levels
- **Critical** (75-100): Immediate capital raise very likely
- **High** (60-74): Strong likelihood of capital raise in next 12 months
- **Medium** (40-59): Moderate risk, monitor closely
- **Low** (0-39): No immediate capital raise pressure

### Threshold
- **Score > 50**: Company flagged for monitoring
- **Score ≤ 50**: Below concern threshold

### Key Signals
1. **Cash Runway** (0-40 pts): Most important. < 12 months is concerning.
2. **Liquidity Stress** (0-20 pts): Current ratio < 1.0 is critical.
3. **Debt Maturity** (0-15 pts): Large obligations due in 6-18 months.
4. **Operational Red Flags** (0-15 pts): Revenue decline, negative FCF.
5. **Market & Behavioral** (0-10 pts): Stock down, insider selling.
